In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from backend.agents.ai_sales_coaching.utils import calculate_call_cost

# Example usage with your data:
result = calculate_call_cost(
    call_duration_seconds=1*8*60*60, # agent, hour, minute, second
    time_window_seconds=8,
    rolling_interval_seconds=2,
    cost_per_extraction=0.001858395  # From your extractor example
)

print(f"Total extractions: {result['num_extractions']}")
print(f"Total cost: {result['total_cost']:.6f}")


Total extractions: 14397
Total cost: 26.755313


In [3]:
from backend.agents.extractor import Extractor
from backend.llms.ollama import OllamaLLM, OpenAIOutputMessage, LlamaOutputMessage
from backend.llms.bedrock import BedrockNova
import logging
from backend.utils import setup_logger
from backend.prompt_hub import PromptHub

setup_logger(logging.DEBUG)

openai_llm = OllamaLLM(model_id="gpt-oss:20b", OutputMessage=OpenAIOutputMessage)
llama_llm = OllamaLLM(model_id="llama3.2-vision:11b", OutputMessage=LlamaOutputMessage)
nova_llm = BedrockNova(model_id="us.amazon.nova-micro-v1:0")

✅ Development logging enabled (DEBUG level) - Console


In [4]:
from backend.agents.ai_sales_coaching.extract_data_model import CustomerInfo, CustomerInterest, AgentCheckList

In [5]:
demographic_extractor = Extractor(
    agent_name="demographic_extractor",
    llm=nova_llm,
    system_prompt=PromptHub().extract_customer_information,
    DataModel=CustomerInfo,
)
interest_extractor = Extractor(
    agent_name="interest_extractor",
    llm=nova_llm,
    system_prompt=PromptHub().extract_customer_interest,
    DataModel=CustomerInterest,
)
agent_checklist_extractor = Extractor(
    agent_name="agent_checklist_extractor",
    llm=nova_llm,
    system_prompt=PromptHub().extract_agent_checklist,
    DataModel=AgentCheckList,
)

In [8]:
from backend.llms.utils import parse_blockcode
test = """\
```json
{
  "age": 35,
  "income_per_month": null,
  "marital_status": null,
  "number_of_children": null
}
```
"""
parse_blockcode(test, "json")

'{\n  "age": 35,\n  "income_per_month": null,\n  "marital_status": null,\n  "number_of_children": null\n}'

In [6]:
content = """\
TEXT: สวัสดีครับผมอนันต์จากบริษัทประกันภัย เรียนสายคุยสมชาย ไม่ทราบว่าสะดวกคุยไหมครับ? สะดวกครับ ผมอยากรบกวนสอบถามว่าตอนนี้คุณสมชายอายุเท่าไหร่แล้วครับ ตอนนี้ผมอายุสามสิบห้าปีแล้วครับ
""".strip()
response = demographic_extractor.run([{"role": "user", "content": content}])
response

2025-11-26 00:20:58,383 - demographic_extractor - run:67 - INFO - start extracting...
2025-11-26 00:20:58,384 - demographic_extractor - run:71 - DEBUG - original_messages: [{'role': 'user', 'content': 'TEXT: สวัสดีครับผมอนันต์จากบริษัทประกันภัย เรียนสายคุยสมชาย ไม่ทราบว่าสะดวกคุยไหมครับ? สะดวกครับ ผมอยากรบกวนสอบถามว่าตอนนี้คุณสมชายอายุเท่าไหร่แล้วครับ ตอนนี้ผมอายุสามสิบห้าปีแล้วครับ'}]
2025-11-26 00:20:59,749 - demographic_extractor - _run:49 - DEBUG - LLM raw output: ```json
{
  "age": 35,
  "income_per_month": null,
  "marital_status": null,
  "number_of_children": null
}
```
2025-11-26 00:20:59,750 - demographic_extractor - _run:53 - DEBUG - Parsed output: 
2025-11-26 00:20:59,750 - demographic_extractor - run:79 - WARNING - attempt 1 failed: Empty input
2025-11-26 00:20:59,750 - demographic_extractor - run:96 - INFO - end extracting with input_tokens=527 and output_tokens=42
2025-11-26 00:21:01,560 - demographic_extractor - _run:49 - DEBUG - LLM raw output: TEXT: สวัสดีครับผมอนันต์

CustomerInfo(age=None, income_per_month=None, marital_status=None, number_of_children=None)

In [9]:
content = """\
TEXT: ไม่ทราบว่าคุณสมชายมีเป้าหมายทางการเงินอะไรบ้างครับ? ผมอยากมีเงินใช้หลังเกษียนซักประมาณสามหมื่นบาทต่อเดือนน่ะครับ
""".strip()
response = interest_extractor.run([{"role": "user", "content": content}])
response

2025-11-26 00:16:54,099 - interest_extractor - run:65 - INFO - start extracting...
2025-11-26 00:16:55,605 - interest_extractor - run:75 - WARNING - attempt 1 failed: Empty input
2025-11-26 00:16:55,606 - interest_extractor - run:92 - INFO - end extracting with input_tokens=497 and output_tokens=42
2025-11-26 00:16:57,093 - interest_extractor - run:72 - INFO - extraction successful on attempt 2
2025-11-26 00:16:57,093 - interest_extractor - run:92 - INFO - end extracting with input_tokens=1019 and output_tokens=110


CustomerInterest(life_insurance=None, health_insurance=None, critical_illness=None, accident_insurance=None, retirement_planning=None, tax_benefits=None)

In [10]:
content = """\
สวัสดีครับผมอนันต์จากบริษัทประกันครับขออนุญาติเรียนสายคุณสมชายเพื่อแนะนำแบบประกันครับ
""".strip()
response = agent_checklist_extractor.run([{"role": "user", "content": content}])
response

2025-11-26 00:17:13,343 - agent_checklist_extractor - run:65 - INFO - start extracting...
2025-11-26 00:17:14,560 - agent_checklist_extractor - run:72 - INFO - extraction successful on attempt 1
2025-11-26 00:17:14,561 - agent_checklist_extractor - run:92 - INFO - end extracting with input_tokens=345 and output_tokens=22


AgentCheckList(agent_introduced=True, company_mentioned=True, permission_asked=True)

In [22]:
demographic_extractor.get_usage_stats()

{'agent_name': 'demographic_extractor',
 'model_id': 'us.amazon.nova-micro-v1:0',
 'input_tokens': 364,
 'output_tokens': 18,
 'input_cost': 0.00042041999999999996,
 'output_cost': 8.316e-05,
 'total_cost': 0.00050358}

In [23]:
interest_extractor.get_usage_stats()

{'agent_name': 'interest_extractor',
 'model_id': 'us.amazon.nova-micro-v1:0',
 'input_tokens': 574,
 'output_tokens': 23,
 'input_cost': 0.0006629699999999999,
 'output_cost': 0.00010626,
 'total_cost': 0.0007692299999999999}

In [24]:
agent_checklist_extractor.get_usage_stats()

{'agent_name': 'agent_checklist_extractor',
 'model_id': 'us.amazon.nova-micro-v1:0',
 'input_tokens': 357,
 'output_tokens': 22,
 'input_cost': 0.000412335,
 'output_cost': 0.00010164,
 'total_cost': 0.000513975}

In [37]:
from backend.prompt_hub import PromptHub

In [38]:
prompt_str = PromptHub().classify_sales_stage
print(prompt_str)

# PERSONA

You are an expert sales conversation stage classifier for real-time telesales analysis.

# CONTEXT

You will receive partial conversation transcripts from ongoing sales calls. The conversation may be incomplete or fragmented. Classify the current stage based on available evidence.

# INSTRUCTION

- Read TEXT carefully (this is an 8-second audio chunk from ongoing conversation)
- Classify the conversation stage based on goals and context
- Extract signals that support your classification
- Do NOT guess - use only clear evidence from the text

# SALES STAGES

**Greeting**: 
	- Goals are to gain permission to talk, build initial trust and warmth, confirm customer identity, set expectations for call length
**Discovery**: 
	- Goals are to understand customer's needs, identify pain points, collect qualifying data, clarify expectations and priorities
**Pitch**: 
	- Goals are to match product to customer needs, highlight benefits using customer language, show quantified value
**Clos

In [6]:
# Create a sales stage classifier
from backend.agents.ai_sales_coaching.extract_data_model import ClassifiedStage
stage_classifier = Extractor(
    agent_name="stage_classifier",
    llm=nova_llm,
    system_prompt=PromptHub().classify_sales_stage,
    DataModel=ClassifiedStage,
    format="json",
)

# Test with different conversation examples
# test_conversations = [
#     "สวัสดีครับผมอนันต์จากบริษัทประกันภัย เรียนสายคุยสมชาย ไม่ทราบว่าสะดวกคุยไหมครับ?",
#     "ไม่ทราบว่าคุณสมชายมีเป้าหมายทางการเงินอะไรบ้างครับ?",
#     "เรามีแผนประกันที่เหมาะกับคุณมาก ให้ผลตอบแทน 6% ต่อปี",
#     "คุณพร้อมที่จะเริ่มต้นแผนนี้ไหมครับ?"
# ]

# for conversation in test_conversations:
#     result = stage_classifier.run([{"role": "user", "content": f"TEXT: {conversation}"}])
#     print(f"Conversation: {conversation[:50]}...")
#     print(f"Stage: {result.stage}")
#     print(f"Signals: {result.signals}")
#     print("---")
# response = nova_llm.run(
#     system_prompt=PromptHub().classify_sales_stage,
#     messages=[dict(role="user", content=test_conversations[0])]
# )

In [7]:
conversation = """\
สวัสดีครับผมอนันต์จากบริษัทประกันชีวิต เรียนสายคุณสมชาย พอจะสะดวกคุยซักห้านาทีไหมครับ? สะดวกครับ
""".strip()

response = stage_classifier.run([{"role": "user", "content": f"TEXT: {conversation}"}])
print(response)
print(stage_classifier.calculate_cost())

2025-11-23 23:51:18,210 - stage_classifier - run:65 - INFO - start extracting...
2025-11-23 23:51:20,904 - stage_classifier - run:72 - INFO - extraction successful on attempt 1
2025-11-23 23:51:20,905 - stage_classifier - run:92 - INFO - end extracting with input_tokens=706 and output_tokens=106


stage=<StageType.GREETING: 'Greeting'> signals=["agent introduction ('ผมอนันต์จากบริษัทประกันชีวิต')", "company mention ('บริษัทประกันชีวิต')", "permission request ('พอจะสะดวกคุยซักห้านาทีไหมครับ')"]
{'input_tokens': 706, 'output_tokens': 106, 'input_cost': 0.00081543, 'output_cost': 0.00048972, 'total_cost': 0.0013051500000000001}


In [8]:
conversation = """\
สะดวกคุยซักห้านาทีไหมครับ? สะดวกครับมีเรื่องอะไรหรือครับ? ผมขอสอบถามพี่สมชายว่าไม่แน่ใจว่าตอนนี้
""".strip()

response = stage_classifier.run([{"role": "user", "content": f"TEXT: {conversation}"}])
print(response)
print(stage_classifier.calculate_cost())

2025-11-23 23:51:20,919 - stage_classifier - run:65 - INFO - start extracting...
2025-11-23 23:51:22,329 - stage_classifier - run:72 - INFO - extraction successful on attempt 1
2025-11-23 23:51:22,330 - stage_classifier - run:92 - INFO - end extracting with input_tokens=704 and output_tokens=52


stage=<StageType.GREETING: 'Greeting'> signals=['requesting permission to talk', 'asking if the customer is available', "agent introduction ('พี่สมชาย')"]
{'input_tokens': 704, 'output_tokens': 52, 'input_cost': 0.00081312, 'output_cost': 0.00024024, 'total_cost': 0.00105336}


In [9]:
conversation = """\
ไม่แน่ใจว่าตอนนี้พี่สมชายสนใจประกันสุขภาพหรือประกันโรคร้ายแรงอยู่ไหมครับ? อืมตอนนี้ผมสนใจแต่ประกันสุขภาพครับ
""".strip()

response = stage_classifier.run([{"role": "user", "content": f"TEXT: {conversation}"}])
print(response)
print(stage_classifier.calculate_cost())

2025-11-23 23:51:22,342 - stage_classifier - run:65 - INFO - start extracting...
2025-11-23 23:51:23,637 - stage_classifier - run:72 - INFO - extraction successful on attempt 1
2025-11-23 23:51:23,638 - stage_classifier - run:92 - INFO - end extracting with input_tokens=713 and output_tokens=46


stage=<StageType.DISCOVERY: 'Discovery'> signals=['asking about specific interest in health insurance or critical illness insurance', 'customer response indicating interest in health insurance']
{'input_tokens': 713, 'output_tokens': 46, 'input_cost': 0.0008235149999999999, 'output_cost': 0.00021252, 'total_cost': 0.0010360349999999998}


In [10]:
conversation = """\
อืมตอนนี้ผมสนใจแต่ประกันสุขภาพครับ ถ้าพี่สมชายสนใจประกันสุขภาพผมแนะนำแผนประกันตัวนี้ครับคุ้มครองในตัวของสุขภาพ ในวงเงินอยู่ที่ห้าพันบาทต่อการรักษาและถ้าในกรณีที่ต้องนอนโรงพยาบาลจะมีค่าห้องให้คืนละห้าพันบาท
""".strip()

response = stage_classifier.run([{"role": "user", "content": f"TEXT: {conversation}"}])
print(response)
print(stage_classifier.calculate_cost())

2025-11-23 23:51:23,653 - stage_classifier - run:65 - INFO - start extracting...
2025-11-23 23:51:26,260 - stage_classifier - run:72 - INFO - extraction successful on attempt 1
2025-11-23 23:51:26,261 - stage_classifier - run:92 - INFO - end extracting with input_tokens=789 and output_tokens=64


stage=<StageType.PITCH: 'Pitch'> signals=['mentioning specific insurance plan', 'highlighting benefits and coverage', 'showing quantified value (5,000 baht for treatment and 5,000 baht for hospital room)']
{'input_tokens': 789, 'output_tokens': 64, 'input_cost': 0.000911295, 'output_cost': 0.00029568, 'total_cost': 0.0012069749999999999}


In [11]:
conversation = """\
นอนโรงพยาบาลจะมีค่าห้องให้คืนละห้าพันบาท ไม่แน่ใจว่าใช่ที่พี่สมชายมองหาไหมครับ? ค่ารักษาในแต่ละครั้งห้าพันบาทนี่มันครอบคลุมโรคอะไรบ้างครับ?
""".strip()

response = stage_classifier.run([{"role": "user", "content": f"TEXT: {conversation}"}])
print(response)
print(stage_classifier.calculate_cost())

2025-11-23 23:51:26,276 - stage_classifier - run:65 - INFO - start extracting...
2025-11-23 23:51:28,060 - stage_classifier - run:72 - INFO - extraction successful on attempt 1
2025-11-23 23:51:28,061 - stage_classifier - run:92 - INFO - end extracting with input_tokens=739 and output_tokens=56


stage=<StageType.DISCOVERY: 'Discovery'> signals=['asking about specific details of hospital room refund', 'inquiring about coverage of treatment costs', 'collecting qualifying data on what conditions are covered']
{'input_tokens': 739, 'output_tokens': 56, 'input_cost': 0.0008535449999999999, 'output_cost': 0.00025872, 'total_cost': 0.0011122649999999999}


In [12]:
conversation = """\
ครั้งห้าพันบาทนี่มันครอบคลุมโรคอะไรบ้างครับ? ในกรณีค่ารักษาอันนี้ครอบคลุมได้หมดทุกโรคเลยครับพี่สมชาย ยกเว้น อุบัติเหตุเท่านั้น
""".strip()

response = stage_classifier.run([{"role": "user", "content": f"TEXT: {conversation}"}])
print(response)
print(stage_classifier.calculate_cost())

2025-11-23 23:51:28,076 - stage_classifier - run:65 - INFO - start extracting...
2025-11-23 23:51:29,504 - stage_classifier - run:72 - INFO - extraction successful on attempt 1
2025-11-23 23:51:29,504 - stage_classifier - run:92 - INFO - end extracting with input_tokens=731 and output_tokens=52


stage=<StageType.DISCOVERY: 'Discovery'> signals=['asking about coverage details', 'inquiring about specific conditions covered', 'requesting information on what is included in the plan']
{'input_tokens': 731, 'output_tokens': 52, 'input_cost': 0.000844305, 'output_cost': 0.00024024, 'total_cost': 0.001084545}


In [13]:
conversation = """\
เลยครับพี่สมชาย ยกเว้น อุบัติเหตุเท่านั้น น่าสนใจอยู่นะ ไม่แน่ใจว่าอันนี้ค่าเบี้ยประกันเป็นยังไงบ้าง?
""".strip()

response = stage_classifier.run([{"role": "user", "content": f"TEXT: {conversation}"}])
print(response)
print(stage_classifier.calculate_cost())


2025-11-23 23:51:29,515 - stage_classifier - run:65 - INFO - start extracting...
2025-11-23 23:51:30,909 - stage_classifier - run:72 - INFO - extraction successful on attempt 1
2025-11-23 23:51:30,912 - stage_classifier - run:92 - INFO - end extracting with input_tokens=707 and output_tokens=38


stage=<StageType.DISCOVERY: 'Discovery'> signals=['asking about insurance premium', 'showing interest in the product']
{'input_tokens': 707, 'output_tokens': 38, 'input_cost': 0.000816585, 'output_cost': 0.00017556, 'total_cost': 0.000992145}


In [14]:
conversation = """\
้ค่าเบี้ยประกันเป็นยังไงบ้าง? ค่าเบี้ยประกันตอนตัวนี้อยู่ที่เดือนละห้าร้อยบาทครับ ไม่แน่ใจว่าพี่สมชายสนใจไหม? โอน่าสนใจ
""".strip()

response = stage_classifier.run([{"role": "user", "content": f"TEXT: {conversation}"}])
print(response)
print(stage_classifier.calculate_cost())

2025-11-23 23:51:30,927 - stage_classifier - run:65 - INFO - start extracting...
2025-11-23 23:51:32,322 - stage_classifier - run:72 - INFO - extraction successful on attempt 1
2025-11-23 23:51:32,323 - stage_classifier - run:92 - INFO - end extracting with input_tokens=717 and output_tokens=43


stage=<StageType.DISCOVERY: 'Discovery'> signals=['asking about insurance premium', 'providing specific product details', 'indicating potential interest']
{'input_tokens': 717, 'output_tokens': 43, 'input_cost': 0.0008281349999999999, 'output_cost': 0.00019866, 'total_cost': 0.0010267949999999998}


In [15]:
conversation = """\
ใจว่าพี่สมชายสนใจไหม? โอน่าสนใจถ้างั้นผมต้องทำยังไงบ้าง? ถ้าพี่สมชายสนใจงั้นผมคุยรายละเอียดเรื่องการสมัครเลยไหมครับ?
""".strip()

response = stage_classifier.run([{"role": "user", "content": f"TEXT: {conversation}"}])
print(response)
print(stage_classifier.calculate_cost())

2025-11-23 23:51:32,338 - stage_classifier - run:65 - INFO - start extracting...
2025-11-23 23:51:34,768 - stage_classifier - run:72 - INFO - extraction successful on attempt 1
2025-11-23 23:51:34,769 - stage_classifier - run:92 - INFO - end extracting with input_tokens=720 and output_tokens=53


stage=<StageType.CLOSING: 'Closing'> signals=['asking if the customer is interested', 'requesting details on how to proceed if interested', 'inviting to discuss enrollment details']
{'input_tokens': 720, 'output_tokens': 53, 'input_cost': 0.0008315999999999999, 'output_cost': 0.00024486, 'total_cost': 0.00107646}


In [45]:
from backend.agents.ai_sales_coaching.coaching_data_model import SalesCoaching

# Create sales coach
sales_coach = Extractor(
    agent_name="sales_coach",
    llm=nova_llm,
    # llm=openai_llm,
    system_prompt=PromptHub().sales_coaching,
    DataModel=SalesCoaching,
    format="json"
)

# Example usage
# conversation_context = """
# Current Stage: Discovery
# Recent Text: "คุณมีครอบครัวกี่คนครับ? มีสองคนครับ ผมกับภรรยา"
# Customer Data: {"age": 35, "family_size": 2}
# """

conversation_context = """\
Current Stage: Discovery 
Recent Text: "ไม่แน่ใจว่าตอนนี้พี่สมชายสนใจประกันสุขภาพหรือประกันโรคร้ายแรงอยู่ไหมครับ? อืมตอนนี้ผมสนใจแต่ประกันสุขภาพครับ"
""".strip()


conversation_context = """\
Current Stage: Discovery 
Recent Text: "นอนโรงพยาบาลจะมีค่าห้องให้คืนละห้าพันบาท ไม่แน่ใจว่าใช่ที่พี่สมชายมองหาไหมครับ? ค่ารักษาในแต่ละครั้งห้าพันบาทนี่มันครอบคลุมโรคอะไรบ้างครับ?"
""".strip()

conversation_context = """\
Current Stage: Discovery 
Recent Text: "ครั้งห้าพันบาทนี่มันครอบคลุมโรคอะไรบ้างครับ? ในกรณีค่ารักษาอันนี้ครอบคลุมได้หมดทุกโรคเลยครับพี่สมชาย ยกเว้น อุบัติเหตุเท่านั้น"
""".strip()

conversation_context = """\
Current Stage: Discovery 
Recent Text: "ครอบคลุมได้หมดทุกโรคเลยครับพี่สมชาย ยกเว้น อุบัติเหตุเท่านั้น น่าสนใจอยู่นะ ไม่แน่ใจว่าอันนี้ค่าเบี้ยประกันเป็นยังไงบ้าง?"
""".strip()

conversation_context = """\
Current Stage: Discovery 
Recent Text: "ค่าเบี้ยประกันเป็นยังไงบ้าง? ค่าเบี้ยประกันตอนตัวนี้อยู่ที่เดือนละห้าร้อยบาทครับ ไม่แน่ใจว่าพี่สมชายสนใจไหม? โอน่าสนใจ"
""".strip()
conversation_context = """\
Current Stage: Discovery 
Recent Text: "ใจว่าพี่สมชายสนใจไหม? โอน่าสนใจถ้างั้นผมต้องทำยังไงบ้าง? ถ้าพี่สมชายสนใจงั้นผมคุยรายละเอียดเรื่องการสมัครเลยไหมครับ?"
""".strip()
response = sales_coach.run([{"role": "user", "content": conversation_context}])
response
# print(f"Next Action: {coaching.next_action}")
# print(f"Suggested Lines: {coaching.suggested_lines}")
# print(f"Stage Transition: {coaching.stage_transition}")


2025-11-24 00:14:51,739 - sales_coach - run:65 - INFO - start extracting...
2025-11-24 00:14:56,659 - sales_coach - run:72 - INFO - extraction successful on attempt 1
2025-11-24 00:14:56,660 - sales_coach - run:92 - INFO - end extracting with input_tokens=615 and output_tokens=236


SalesCoaching(current_stage='Discovery', next_action='ask_about_specific_needs', suggested_lines=['พี่สมชาย คุณมีความต้องการเฉพาะอย่างใดอย่างหนึ่งที่เราสามารถช่วยแก้ไขได้หรือไม่ครับ? เช่น การปกป้องครอบครัวหรือการดูแลสุขภาพ?', 'คุณมีความกังวลเกี่ยวกับอายุการเกษียณหรือการดูแลความคุ้มครองในอนาคตหรือไม่ครับ?'], stage_transition=<StageTransition.STAY: 'stay'>, reason='The customer is showing interest but lacks specific needs. Clarifying their specific needs will help tailor the pitch to their requirements.')

In [41]:
sales_coach.calculate_cost()

{'input_tokens': 627,
 'output_tokens': 178,
 'input_cost': 0.000724185,
 'output_cost': 0.00082236,
 'total_cost': 0.0015465449999999999}

In [19]:
response = openai_llm.run(PromptHub().sales_coaching, [{"role": "user", "content": conversation_context}])
print(response.content)

```json
{
  "current_stage": "Discovery",
  "next_action": "ask_about_budget",
  "suggested_lines": [
    "ขอให้บอกหน่อยว่า งบประมาณที่คุณสะดวกต่อการซื้อประกันต่อเดือนประมาณเท่าไหร่ครับ?",
    "ถ้าตั้งใจจะมีการชำระค่างวดประกันอย่างสม่ำเสมอ คุณคาดว่าจะใช้จำนวนเงินเท่าไหร่ต่อเดือน?"
  ],
  "stage_transition": "stay",
  "reason": "ยังขาดข้อมูลสำคัญเกี่ยวกับงบประมาณการซื้อประกัน เพื่อให้สามารถนำเสนอแผนที่เหมาะสมกับลูกค้าได้"
}
```


In [32]:
response = nova_llm.run(PromptHub().sales_coaching, [{"role": "user", "content": conversation_context}])
print(response.content)

```json
{
  "current_stage": "Discovery",
  "next_action": "ask_about_insurance_needs",
  "suggested_lines": ["จากที่ทราบแล้วว่าคุณมีภรรยาและลูกสองคน คุณมีความต้องการเกี่ยวกับประกันชีวิตหรือประกันสุขภาพใด ๆ อย่างเฉพาะอย่างไหมครับ?"],
  "stage_transition": "stay",
  "reason": "Need to understand specific insurance needs to tailor the pitch effectively"
}
```


In [33]:
from backend.llms.utils import parse_blockcode
import json
parsed_resposne = parse_blockcode(response.content, "json")
parsed_response = json.loads(parsed_resposne)

In [34]:
print(parsed_resposne)

{
  "current_stage": "Discovery",
  "next_action": "ask_about_insurance_needs",
  "suggested_lines": ["จากที่ทราบแล้วว่าคุณมีภรรยาและลูกสองคน คุณมีความต้องการเกี่ยวกับประกันชีวิตหรือประกันสุขภาพใด ๆ อย่างเฉพาะอย่างไหมครับ?"],
  "stage_transition": "stay",
  "reason": "Need to understand specific insurance needs to tailor the pitch effectively"
}


In [ ]:
SalesCoaching(**parsed_response)[]

SalesCoaching(current_stage='Discovery', next_action='ask_about_insurance_needs', suggested_lines=['จากที่ทราบแล้วว่าคุณมีภรรยาและลูกสองคน คุณมีความต้องการเกี่ยวกับประกันชีวิตหรือประกันสุขภาพใด ๆ อย่างเฉพาะอย่างไหมครับ?'], stage_transition=<StageTransition.STAY: 'stay'>, reason='Need to understand specific insurance needs to tailor the pitch effectively')

In [18]:
from backend.agents.extractor import Extractor
from backend.prompt_hub import PromptHub
from backend.llms.bedrock import BedrockNova
from backend.agents.ai_sales_coaching.extract_data_model import AgentCheckList

nova_llm = BedrockNova(model_id="us.amazon.nova-micro-v1:0")

agent_checklist_extractor = Extractor(
    agent_name="agent_checklist_extractor",
    llm=nova_llm,
    system_prompt=PromptHub().extract_agent_checklist,
    DataModel=AgentCheckList,
)

In [22]:
# content = """\
# สวัสดีครับผมอนันต์จากบริษัทประกันครับขออนุญาติเรียนสายคุณสมชายเพื่อแนะนำแบบประกันครับ
# """.strip()
# content = """\
# สวัสดีครับผมอนันต์เรียนสายคุณสมชายครับ
# """.strip()
content = """\
เรียนสายคุณสมชาย พอจะสะดวกคุยซักห้านาทีไหมครับ? สะดวกครับ
""".strip()

response = agent_checklist_extractor.run([dict(role="user", content=content)])
response

2025-11-24 21:57:25,379 - agent_checklist_extractor - run:64 - INFO - start extracting...
2025-11-24 21:57:26,632 - agent_checklist_extractor - run:71 - INFO - extraction successful on attempt 1
2025-11-24 21:57:26,633 - agent_checklist_extractor - run:91 - INFO - end extracting with input_tokens=327 and output_tokens=22


AgentCheckList(agent_introduced=None, company_mentioned=None, permission_asked=True)

In [1]:
from backend.llms.bedrock import BedrockNova
from backend.prompt_hub import PromptHub
from backend.agents.extractor import Extractor
from backend.agents.ai_sales_coaching.extract_data_model import Guide

nova_llm = BedrockNova(model_id="us.amazon.nova-micro-v1:0")

In [9]:
from backend.agents.ai_sales_coaching.extract_data_model import Guide

# 1. GREETING AGENT
greeting_agent = Extractor(
    agent_name="greeting_agent",
    llm=nova_llm,
    system_prompt=PromptHub().greeting_agent,
    DataModel=Guide,
    format="json",
)

content = """\
TEXT: สวัสดีครับผมอนันต์จากบริษัทประกันภัยครับ เรียนสายคุณสมชายครับ ไม่ทราบว่าสะดวกคุยไหมครับ? อ๋อ รีบอยู่นะครับ มีแค่ 2-3 นาทีเท่านั้น
""".strip()

response = greeting_agent.run([dict(role="user", content=content)])
print("GREETING AGENT:")
print(response.model_dump_json(indent=4))
greeting_agent.calculate_cost()

GREETING AGENT:
{
    "action": "Ask for brief permission and set time expectation",
    "explanation": "The customer expressed time pressure and gave permission to speak but also indicated a brief time frame. The agent should confirm the time constraint and assure the customer that the conversation will be brief.",
    "signals": [
        "Customer expressed time pressure",
        "Permission granted"
    ],
    "lines_to_say": [
        "ขอบคุณที่รับสาย คุณสมชาย เพราะเวลาของคุณนั้นสำคัญ ฉันจะเริ่มทันทีด้วยเวลาที่เหลือของคุณ 2-3 นาทีนี้ อยากบอกว่าในเรื่องประกันภัยมีสิ่งใหม่ๆ ที่อาจช่วยให้คุณรู้ได้"
    ]
}


{'input_tokens': 579,
 'output_tokens': 229,
 'input_cost': 0.000668745,
 'output_cost': 0.00105798,
 'total_cost': 0.0017267250000000001}

In [10]:
# 2. DISCOVERY AGENT
discovery_agent = Extractor(
    agent_name="discovery_agent",
    llm=nova_llm,
    system_prompt=PromptHub().discovery_agent,
    DataModel=Guide,
    format="json",
)

content = """\
TEXT: ไม่ทราบว่าคุณสมชายมีครอบครัวไหมครับ? มีครับ มีภรรยาแล้วก็ลูกสองคนครับ ลูกชายกับลูกสาว อายุ 8 กับ 5 ขวบครับ แล้วตอนนี้คุณมีประกันอะไรอยู่บ้างไหมครับ? มีประกันสังคมจากที่ทำงานครับ แต่รู้สึกว่ามันไม่เพียงพอน่ะครับ ถ้าเกิดอะไรขึ้นกับผม ครอบครัวจะอยู่ยังไงครับ
""".strip()

response = discovery_agent.run([dict(role="user", content=content)])
print(response.model_dump_json(indent=4))
discovery_agent.calculate_cost()

{
    "action": "Explore family protection needs",
    "explanation": "The customer has mentioned having a family, which indicates a need for family protection. It's important to delve deeper into their specific concerns and how they perceive the adequacy of their current coverage.",
    "signals": [
        "Customer revealed family situation",
        "Customer expressed concern about current coverage"
    ],
    "lines_to_say": [
        "นั่นแหละ คุณมีภรรยาและลูกสองคน การคุ้มครองครอบครัวของคุณเป็นสิ่งที่สำคัญมาก คุณมีความกังวลอย่างไรเกี่ยวกับประกันสังคมที่คุณมีอยู่ในปัจจุบัน?",
        "คุณมีความกังวลว่าหากเกิดเหตุการณ์ฉุกเฉินอะไรขึ้นกับคุณ ครอบครัวของคุณจะอยู่ได้อย่างไรหรือไม่?"
    ]
}


{'input_tokens': 642,
 'output_tokens': 281,
 'input_cost': 0.00074151,
 'output_cost': 0.0012982199999999999,
 'total_cost': 0.00203973}

In [11]:
# 3. PITCH AGENT  
pitch_agent = Extractor(
    agent_name="pitch_agent",
    llm=nova_llm,
    system_prompt=PromptHub().pitch_agent,
    DataModel=Guide,
    format="json",
)

content = """\
TEXT: ตามที่คุณสมชายบอกว่าห่วงเรื่องครอบครัว ผมเลยอยากเสนอแบบประกันชีวิตที่จะช่วยดูแลครอบครัวคุณได้ครับ มีความคุ้มครองสูงสุด 5 ล้านบาท เบี้ยประกันเดือนละ 3,500 บาท ไม่เข้าใจครับ ความคุ้มครองหมายถึงอะไร?
""".strip()

response = pitch_agent.run([dict(role="user", content=content)])
print("\nPITCH AGENT:")
print(response.model_dump_json(indent=4))
pitch_agent.calculate_cost()


PITCH AGENT:
{
    "action": "Clarify the coverage and highlight benefits using customer language",
    "explanation": "The customer expressed confusion about the coverage. The agent should clarify what the coverage means and highlight the benefits in a way that resonates with the customer's concern for family care.",
    "signals": [
        "Customer confused about coverage",
        "Customer expressed interest in family care"
    ],
    "lines_to_say": [
        "คุณกล่าวว่าห่วงเรื่องครอบครัว ดังนั้นผมอยากเล่าให้คุณฟังว่า ความคุ้มครองนี้หมายถึงการปกป้องครอบครัวของคุณในกรณีที่เกิดเหตุเลวร้าย เช่น การเสียชีวิตหรือเกิดอุบัติเหตุที่ทำให้คุณไม่สามารถดูแลครอบครัวได้",
        "ด้วยประกันชีวิตนี้คุณจะได้รับเงินประกันสูงสุดถึง 5 ล้านบาท ซึ่งจะช่วยให้ครอบครัวของคุณมีความมั่นคงทางเศรษฐีในสถานการณ์ที่ไม่คาดคิด"
    ]
}


{'input_tokens': 591,
 'output_tokens': 376,
 'input_cost': 0.000682605,
 'output_cost': 0.00173712,
 'total_cost': 0.002419725}

In [12]:
# 4. CLOSING AGENT
closing_agent = Extractor(
    agent_name="closing_agent", 
    llm=nova_llm,
    system_prompt=PromptHub().closing_agent,
    DataModel=Guide,
    format="json",
)

content = """\
TEXT: ถ้าอย่างนั้นคุณสมชายพร้อมที่จะสมัครแบบประกันนี้ไหมครับ? ผมสนใจครับ แต่ขอคุยกับภรรยาก่อนได้ไหม? เพราะเป็นเรื่องใหญ่สำหรับครอบครัว
""".strip()

response = closing_agent.run([dict(role="user", content=content)])
print("\nCLOSING AGENT:")
print(response.model_dump_json(indent=4))
closing_agent.calculate_cost()


CLOSING AGENT:
{
    "action": "Offer to include family in discussion",
    "explanation": "The customer expresses interest but requests to discuss with his spouse first. It's important to respect this and offer a way to include the family in the decision-making process.",
    "signals": [
        "Customer needs family consultation",
        "Customer expresses interest but has a request"
    ],
    "lines_to_say": [
        "แน่นอน คุณสามารถคุยกับภรรยาได้ เราสามารถจัดเวลาพบเจอกับคุณและภรรยาเพื่อให้คุณสามารถแบ่งปันข้อมูลและตัดสินใจร่วมกันได้",
        "คุณสามารถแจ้งให้ภรรยาของคุณทราบว่าเราพร้อมรับสมัครพร้อมกันได้"
    ]
}


{'input_tokens': 538,
 'output_tokens': 241,
 'input_cost': 0.00062139,
 'output_cost': 0.00111342,
 'total_cost': 0.0017348099999999998}

In [13]:
# 5. OBJECTION HANDLING AGENT
objection_handling_agent = Extractor(
    agent_name="objection_handling_agent",
    llm=nova_llm, 
    system_prompt=PromptHub().objection_handling_agent,
    DataModel=Guide,
    format="json",
)

content = """\
TEXT: เบี้ยประกันเดือนละ 3,500 บาท แพงไปไหมครับ? เงินเดือนผมแค่ 25,000 บาท ต้องจ่ายค่าใช้จ่ายในบ้านอีกเยอะ ไม่รู้จะเหลือพอไหม
""".strip()

response = objection_handling_agent.run([dict(role="user", content=content)])
print("\nOBJECTION HANDLING AGENT:")
print(response.model_dump_json(indent=4))
objection_handling_agent.calculate_cost()


OBJECTION HANDLING AGENT:
{
    "action": "Acknowledge concern and reframe value",
    "explanation": "The customer is expressing concern about the price being too high relative to their budget. It's important to validate their concern and then reframe the value of the insurance to show how it provides long-term benefits.",
    "signals": [
        "Price objection raised",
        "Customer showing concern about affordability"
    ],
    "lines_to_say": [
        "ผมเข้าใจว่าคุณกังวลเรื่องค่าเบี้ยที่เกินรายได้ของคุณ แต่เรามีแผนที่คุ้มครองคุณในระยะยาวอย่างมีประโยชน์",
        "คุณมีคุณสมบัติที่ดีที่จะได้รับความคุ้มครองในสถานการณ์ที่สำคัญ และเราสามารถช่วยให้คุณคุมค่าได้อย่างมีประสิทธิภาพ"
    ]
}


{'input_tokens': 533,
 'output_tokens': 269,
 'input_cost': 0.000615615,
 'output_cost': 0.00124278,
 'total_cost': 0.001858395}

In [7]:
class Test:
    pass

In [8]:
isinstance(Test(), Test)

True

In [1]:
from backend.agents.ai_sales_coaching.stage_agent import StageAgent
from backend.llms.bedrock import BedrockNova
from backend.websocket_tasks.base import Context

context = Context(stage="discovery")
nova_llm = BedrockNova(model_id="us.amazon.nova-micro-v1:0")

stage_agent = StageAgent(llm=nova_llm)

response = stage_agent.run(messages=[dict(role="user", content="TEXT: สวัสดีครับผมอนันต์จากบริษัทประกันชีวิต เรียนสายคุณสมชาย พอจะสะดวกคุยซักห้านาทีไหมครับ? สะดวกครับ")], context=context)
response

attempt 1 failed: Expecting ',' delimiter: line 5 column 24 (char 339)


Guide(stage='discovery', action='ให้เจ้าหน้าที่ฟังและตอบรับคำขอของลูกค้า', explanation='ลูกค้าขอเวลาพูดคุย เจ้าหน้าที่ควรให้ความสำคัญกับความต้องการของลูกค้าและตอบรับคำขอของลูกค้าด้วยความเคารพ', signals=['ลูกค้าขอเวลาพูดคุย'], lines_to_say=['แน่นอน คุณสามารถพูดคุยกับฉันได้เลย เรามีเวลาพูดคุยกันไปห้านาที'])